# Partie logistique – approfondissement (cours 1 à 4)
Cette partie **s'ajoute** aux cellules précédentes (qui restent inchangées). Elle réutilise les variables déjà chargées :
`bp, facilities, receipts, adjustments, month_end, po_lines, shipments`.
Les anomalies repérées sont des **suspicions à examiner**, pas des fraudes avérées.

## 0. Correctif de nettoyage et préparation (cours 1)
**Constat :** avec pandas ≥ 3 les colonnes texte ont le dtype `str` et non `object`. La condition `dtype == 'object'` de
`clean_numeric_column` est donc fausse et la fonction ne fait rien. On garde la fonction (même logique) en corrigeant seulement cette condition,
puis on l'applique à toutes les colonnes numériques stockées en texte (virgules décimales, unités collées).

In [5]:
import sys
!{sys.executable} -m pip install networkx

  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
Using cached networkx-3.7-py3-none-any.whl (2.1 MB)


In [2]:
import sys
!{sys.executable} -m pip install matplotlib scipy scikit-learn

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   --------- ------------------------------ 2.4/9.5 MB 13.3 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.5 MB 17.2 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 17.5 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ------ --------------------------------- 6.0/37.4 MB 28.5 MB/s eta 0:00:02
   ----------- ---------------------------- 10.7/37.4 MB 24.9 MB/s eta 0:00:02
   ---------------- ----------------------- 15.7/37.4 MB 24.9 MB/s eta 0:00:01
   ---------------------- ----------------- 20.7/37.4 MB 24.9 MB/s eta 0:00:01
   ---------------------------- ----------- 26.5/37.4 MB 25.2 MB/s eta 0:00:01
   ---------------------------------- ----- 32.2/37.4 MB 25.4 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 25.8 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 24.6 MB/s  0:00:01
   --

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import IsolationForest

print("Toutes les bibliothèques sont chargées !")

Toutes les bibliothèques sont chargées !


In [8]:
import os

print("Fichiers contenant 'receipt' :")
for f in files:
    if "facilities" in os.path.basename(f).lower():
        print(f)

Fichiers contenant 'receipt' :
data\logistics\erp_facilities.csv
data\public_records\licensed_facilities.csv


In [9]:
import pandas as pd
import glob
import os

# Recherche des fichiers CSV dans le dossier du projet
files = glob.glob("**/*.csv", recursive=True)

def load_csv(filename):
    path = next((f for f in files if os.path.basename(f) == filename), None)
    if path is None:
        raise FileNotFoundError(f"Fichier introuvable : {filename}")
    return pd.read_csv(path)

# Chargement des datasets
shipments = load_csv("shipments.csv")
receipts = load_csv("goods_receipts.csv")
po_lines = load_csv("purchase_order_lines.csv")
adjustments = load_csv("inventory_adjustments.csv")
month_end = load_csv("inventory_month_end.csv")
bp = load_csv("erp_business_partners.csv")
facilities = load_csv("licensed_facilities.csv")

print("Tous les datasets sont chargés !")

Tous les datasets sont chargés !


In [10]:
print(shipments.shape)
print(receipts.shape)
print(po_lines.shape)
print(adjustments.shape)
print(month_end.shape)
print(bp.shape)
print(facilities.shape)

(452, 16)
(612, 12)
(651, 16)
(140, 11)
(502, 11)
(80, 11)
(3, 12)


In [11]:
import glob, itertools, warnings
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.cluster import DBSCAN
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

def clean_numeric_column(df, column_name):
    # Même logique que la version initiale ; seule la condition change (dtype texte quel qu'il soit)
    if not pd.api.types.is_numeric_dtype(df[column_name]):
        df[column_name] = df[column_name].astype("string").str.replace(",", ".", regex=False)
        df[column_name] = df[column_name].replace(r"[^\d\.\-]", "", regex=True)
        df[column_name] = pd.to_numeric(df[column_name], errors="coerce")
    return df

def norm_text(s):
    """Espaces multiples, espaces de bord et casse (ex. 'DELIVERED' / 'Delivered' / 'delivered')."""
    return s.astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.lower()

def parse_dates(s):
    """Formats mixtes présents dans les fichiers (ISO, dd/mm/yyyy, dd-Mon-yyyy, yyyy/mm/dd...)."""
    s = s.astype("string").str.strip()
    out = pd.to_datetime(s, format="%Y-%m-%dT%H:%M:%S", errors="coerce")
    for f in ["%Y-%m-%d", "%d-%b-%Y %H:%M:%S", "%Y/%m/%d %H:%M:%S", "%d/%m/%Y %H:%M:%S",
              "%d/%m/%Y", "%d-%b-%Y", "%Y/%m/%d", "%m-%d-%Y"]:
        m = out.isna() & s.notna()
        out[m] = pd.to_datetime(s[m], format=f, errors="coerce")
    return out

# --- doublons exacts
shipments = shipments.drop_duplicates().copy()
receipts = receipts.drop_duplicates().copy()
po_lines = po_lines.drop_duplicates(["purchase_order_number", "purchase_order_line"]).copy()   # 1 ligne de commande en double
# --- numériques
for df, cols in [(shipments, ["declared_quantity"]), (receipts, ["received_quantity"]),
                 (adjustments, ["adjustment_quantity"]), (po_lines, ["ordered_quantity", "unit_price"]),
                 (month_end, ["opening_stock", "received_quantity", "outgoing_quantity",
                              "documented_internal_use", "documented_waste", "closing_stock"])]:
    for c in cols:
        clean_numeric_column(df, c)
# --- catégories et espaces
for df, cols in [(shipments, ["declared_purpose", "temperature_control_class", "shipment_status"]),
                 (receipts, ["receipt_status"]), (adjustments, ["adjustment_reason"]),
                 (month_end, ["inventory_status"]), (bp, ["partner_role"])]:
    for c in cols:
        df[c] = norm_text(df[c])
bp["partner_name"] = bp["partner_name"].str.strip().str.replace(r"\s+", " ", regex=True)
# --- dates
shipments["dispatch_time"] = parse_dates(shipments["dispatch_time"]); shipments["arrival_time"] = parse_dates(shipments["arrival_time"])
receipts["receipt_date"] = parse_dates(receipts["receipt_date"])
adjustments["adjustment_date"] = parse_dates(adjustments["adjustment_date"])
po_lines["order_date"] = parse_dates(po_lines["order_date"]); po_lines["requested_delivery_date"] = parse_dates(po_lines["requested_delivery_date"])
month_end["reporting_month"] = parse_dates(month_end["reporting_month"])

shipments["transit_h"] = (shipments["arrival_time"] - shipments["dispatch_time"]).dt.total_seconds() / 3600
print("NaN restants :", {n: int(d.select_dtypes(include=["number"]).isna().sum().sum()) for n, d in
      [("shipments", shipments), ("receipts", receipts), ("adjustments", adjustments), ("month_end", month_end), ("po_lines", po_lines)]})
print("Dates non lues :", int(shipments.dispatch_time.isna().sum() + receipts.receipt_date.isna().sum() + adjustments.adjustment_date.isna().sum()))
print("Expéditions après dédoublonnage :", len(shipments), "| transits négatifs :", int((shipments.transit_h < 0).sum()))

NaN restants : {'shipments': 17, 'receipts': 0, 'adjustments': 0, 'month_end': 0, 'po_lines': 0}
Dates non lues : 0
Expéditions après dédoublonnage : 450 | transits négatifs : 0


**Lecture.** Le jeu est nettoyé : doublons exacts supprimés, quantités en texte converties (`'14 kit'`, `'5,42'`), casse et espaces normalisés, dates à formats mixtes lues (0 date perdue). Les 17 `arrival_time` manquants correspondent à des expéditions encore `in_transit` : ce n'est pas une erreur à imputer.

## 1. Les partenaires « sans registre » sont-ils vraiment particuliers ? (cours 1 : ne pas conclure sans comparer)
On teste si les 14 partenaires sans numéro d'enregistrement reçoivent des catégories, des motifs ou des classes d'autorisation
différentes du reste (test du χ²). Un p-value élevé = pas de différence démontrée.

In [12]:
sans_registre = bp.loc[bp.external_registration_number.isna(), "partner_code"]
shipments["dest_sans_registre"] = shipments.receiver_partner_code.isin(sans_registre)
print(f"Part des expéditions vers ces 14 partenaires : {shipments.dest_sans_registre.mean():.1%} ({shipments.dest_sans_registre.sum()} lignes)")
for col in ["item_category", "declared_purpose", "authorization_class"]:
    chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(shipments[col], shipments.dest_sans_registre))
    print(f"χ² {col:<20} p = {p:.3f}")
shipments["annee"] = shipments.dispatch_time.dt.year
part = shipments.groupby("annee").dest_sans_registre.agg(["sum", "size"]); part["part"] = (part["sum"] / part["size"]).round(2)
print(part.T)

Part des expéditions vers ces 14 partenaires : 16.2% (73 lignes)
χ² item_category        p = 0.188
χ² declared_purpose     p = 0.090
χ² authorization_class  p = 0.197
annee  2014   2015   2016   2017   2018   2019   2020   2021   2022   2023   2024   2025
sum     0.0   1.00   2.00   7.00   6.00   8.00  13.00   5.00   8.00  11.00   7.00   5.00
size    1.0  17.00  17.00  19.00  47.00  45.00  46.00  43.00  61.00  73.00  47.00  34.00
part    0.0   0.06   0.12   0.37   0.13   0.18   0.28   0.12   0.13   0.15   0.15   0.15


**Lecture.** Les 14 partenaires sans registre représentent 16,2 % des expéditions (73 lignes). Leurs catégories d'articles, leurs motifs déclarés et leurs classes d'autorisation **ne diffèrent pas significativement** du reste (p = 0,19 / 0,09 / 0,20). Les pics 2017 (7/19) et 2020 (13/46) reposent sur de petits effectifs.
**Conséquence pour le rapport :** l'absence de registre est une particularité administrative réelle, mais elle ne suffit pas à démontrer un flux atypique. L'hypothèse du notebook initial doit être présentée comme *piste non confirmée* ; les pistes solides sont dans les sections suivantes.

## 2. Motifs fréquents dans les expéditions – algorithme Apriori (cours 3)
Implémentation du pseudo-code du cours (génération de candidats + élagage par anti-monotonie du support). Chaque expédition est une transaction
dont les items sont `colonne=valeur`. Les règles sont évaluées par **support, confidence, lift, leverage**.

In [13]:
ITEM_COLS = ["item_category", "declared_purpose", "authorization_class", "temperature_control_class",
             "origin_facility_code", "destination_facility_code", "sender_partner_code",
             "receiver_partner_code", "carrier_partner_code", "shipment_status"]
transactions = [frozenset(f"{c}={v}" for c, v in row.items()) for row in shipments[ITEM_COLS].astype(str).to_dict("records")]

def apriori_gen(L, k):
    """Candidats de taille k à partir des fréquents de taille k-1, avec élagage."""
    Ls, result = set(L), []
    for p, q in itertools.combinations(sorted(tuple(sorted(x)) for x in L), 2):
        if p[:k-2] == q[:k-2] and p[k-2] < q[k-2]:
            c = frozenset(p) | frozenset(q)
            if all(frozenset(s) in Ls for s in itertools.combinations(c, k - 1)):
                result.append(c)
    return result

def apriori(T, min_support, max_k=4):
    n = len(T); count = {}
    for t in T:
        for i in t: count[frozenset([i])] = count.get(frozenset([i]), 0) + 1
    L = {c: v / n for c, v in count.items() if v / n >= min_support}; frequent = dict(L); k = 2
    while L and k <= max_k:
        counts = {c: 0 for c in apriori_gen(list(L), k)}
        for t in T:
            for c in counts:
                if c <= t: counts[c] += 1
        L = {c: v / n for c, v in counts.items() if v / n >= min_support}; frequent.update(L); k += 1
    return frequent

def association_rules(frequent, min_conf=0.8):
    rows = []
    for itemset, sup in frequent.items():
        if len(itemset) < 2: continue
        for Y in map(frozenset, itertools.combinations(itemset, 1)):       # conséquent à 1 seul item (lisibilité)
            X = itemset - Y
            if len(X) <= 2:
                conf = sup / frequent[X]
                if conf >= min_conf:
                    rows.append({"X": " & ".join(sorted(X)), "Y": " & ".join(sorted(Y)), "support": sup, "confidence": conf,
                                 "lift": conf / frequent[Y], "leverage": sup - frequent[X] * frequent[Y]})
    return pd.DataFrame(rows)

frequent = apriori(transactions, min_support=0.03, max_k=4)
rules = association_rules(frequent, min_conf=0.9)
# on ne garde que les règles qui ont un sens : lift élevé et X qui n'est pas lui-même déterminé par un seul item trivial
top_rules = rules[(rules.lift >= 5)].sort_values(["lift", "support"], ascending=False)
print(f"{len(frequent)} itemsets fréquents, {len(rules)} règles (conf ≥ 0.9, conséquent à 1 item), {len(top_rules)} avec lift ≥ 5")
pd.set_option("display.max_colwidth", 70)
top_rules.drop_duplicates("Y").head(12).round(3)

836 itemsets fréquents, 299 règles (conf ≥ 0.9, conséquent à 1 item), 88 avec lift ≥ 5


,X,Y,support,confidence,lift,leverage
98,carrier_partner_code=BP-30047 & receiver_partner_code=BP-30078,declared_purpose=environmental and wildlife reference materials,0.040,0.900,22.500,0.038
229,destination_facility_code=FCL-1019 & sender_partner_code=BP-30078,receiver_partner_code=BP-30056,0.033,0.938,21.094,0.032
170,destination_facility_code=FCL-1010 & receiver_partner_code=BP-30021,declared_purpose=temperature-controlled clinical programme delivery,0.031,1.000,17.308,0.029
137,carrier_partner_code=BP-30047 & receiver_partner_code=BP-30078,sender_partner_code=BP-30056,0.044,1.000,12.857,0.041
221,destination_facility_code=FCL-1011 & sender_partner_code=BP-30056,origin_facility_code=FCL-1019,0.033,1.000,11.538,0.030
19,receiver_partner_code=BP-30056,destination_facility_code=FCL-1019,0.044,1.000,9.375,0.040
223,receiver_partner_code=BP-30078 & temperature_control_class=ambient,destination_facility_code=FCL-1011,0.071,1.000,9.000,0.063
245,sender_partner_code=BP-30056 & temperature_control_class=2-8c,item_category=biological_sample,0.047,1.000,8.654,0.041
208,receiver_partner_code=BP-30021 & temperature_control_class=ambient,destination_facility_code=FCL-1007,0.056,1.000,8.036,0.049
8,declared_purpose=environmental and wildlife reference materials,receiver_partner_code=BP-30078,0.040,1.000,8.036,0.035


**Lecture.** Apriori fait ressortir des règles très rigides (lift de 5 à 22) qui isolent deux flux :
* transporteur `BP-30047` (Northstar Cold Chain Services) **&** destinataire `BP-30078` (Meridian BioLogistics) ⇒ motif *environmental and wildlife reference materials* (confiance 0,90, lift 22,5) ; ce motif implique toujours l'expéditeur `BP-30056` (Horizon Field Research Cooperative) et du matériel biologique à 2-8 °C ;
* destination `FCL-1010` **&** destinataire `BP-30021` (Asterion Clinical Institute) ⇒ motif *temperature-controlled clinical programme delivery* (confiance 1,0, lift 17,3).
Les autres motifs (« scheduled stock transfer », « customer distribution »…) n'apparaissent dans aucune règle forte : ils sont répartis presque uniformément sur toutes les catégories, ce qui est typique d'un libellé générique.

## 3. Anomalies univariées : IQR et z-score modifié (cours 3)
On construit d'abord l'**écart de stock non expliqué** par la réconciliation de `inventory_month_end` :
`écart = stock_fin − (stock_début + reçu − sorti − usage_documenté − perte_documentée)`.
Un écart ≠ 0 signifie une variation que les quantités documentées n'expliquent pas (écart négatif = « disparition » non justifiée).

In [14]:
month_end["ecart"] = (month_end.closing_stock - (month_end.opening_stock + month_end.received_quantity - month_end.outgoing_quantity
                      - month_end.documented_internal_use - month_end.documented_waste)).astype(float)

def modified_z(s):
    med = s.median(); mad = (s - med).abs().median()
    return 0.6745 * (s - med) / mad if mad > 0 else pd.Series(0.0, index=s.index)

def iqr_flag(s, k=1.5):
    q1, q3 = s.quantile([.25, .75]); return (s < q1 - k * (q3 - q1)) | (s > q3 + k * (q3 - q1))

month_end["z_mod"] = month_end.groupby("item_category")["ecart"].transform(modified_z)   # par catégorie : unités différentes
month_end["iqr"] = month_end.groupby("item_category")["ecart"].transform(iqr_flag)
month_end["aberrant_univarie"] = (month_end.z_mod.abs() > 3.5) & month_end.iqr
print("Écarts de stock aberrants (|z modifié| > 3.5 ET hors IQR) :", int(month_end.aberrant_univarie.sum()), "/", len(month_end))
cols = ["facility_code", "item_category", "reporting_month", "opening_stock", "received_quantity", "outgoing_quantity", "closing_stock", "ecart", "z_mod", "inventory_status"]
print(month_end[month_end.aberrant_univarie].sort_values("z_mod")[cols].round(2).head(12).to_string(index=False))
print("\nPar site :"); print(month_end[month_end.aberrant_univarie].groupby("facility_code").size().sort_values(ascending=False).to_string())
# Quantités expédiées déclarées (z modifié par unité)
shipments["z_qte"] = shipments.groupby("declared_unit")["declared_quantity"].transform(modified_z)
print("\nQuantités expédiées extrêmes (|z modifié| > 3.5) :", int((shipments.z_qte.abs() > 3.5).sum()), "/", len(shipments))

Écarts de stock aberrants (|z modifié| > 3.5 ET hors IQR) : 26 / 502
facility_code        item_category reporting_month  opening_stock  received_quantity  outgoing_quantity  closing_stock  ecart  z_mod inventory_status
     FCL-1027 monitoring_equipment      2021-05-01          24.51              11.49               0.19          16.42 -17.23  -9.97          revised
     FCL-1019 monitoring_equipment      2024-05-01          99.39               9.25                0.0          73.84 -16.69  -9.66          revised
     FCL-1019    biological_sample      2019-11-01          28.43                7.0                7.0           3.38 -21.80  -7.30           closed
     FCL-1010  medical_consumables      2023-11-01          258.7               81.0                0.7         270.62 -17.76  -7.27          revised
     FCL-1024    biological_sample      2018-03-01          16.24                5.0                0.0           0.55 -18.10  -6.08           closed
     FCL-1027 agricultural_prod

**Lecture.** 26 écarts de stock sur 502 sont aberrants selon deux critères combinés (z-score modifié et IQR) ; ils se concentrent sur `FCL-1020` (8), `FCL-1010` (7) et `FCL-1019` (6). Aucune quantité expédiée n'est extrême par unité (0/450), donc l'anomalie n'est pas dans la *taille* des envois mais dans la *comptabilité des stocks*.
**Limite :** un écart non nul existe presque partout (bruit d'arrondi/comptage) ; seuls les écarts très au-delà de la dispersion de leur catégorie sont retenus.

## 4. Anomalies multivariées : Isolation Forest, LOF, DBSCAN (cours 2 et 3)
Variables **standardisées** (cours 2 : les échelles diffèrent selon les unités). On croise les trois méthodes : une ligne n'est retenue
comme suspecte que si **au moins 2 méthodes sur 3** la signalent (limite le risque de faux positifs d'une seule méthode).

In [15]:
FEATS = ["opening_stock", "received_quantity", "outgoing_quantity", "documented_internal_use", "documented_waste", "closing_stock", "ecart"]
# standardisation PAR catégorie d'article (unités et ordres de grandeur différents : pallet, kit, case, sealed_lot)
X = month_end.groupby("item_category")[FEATS].transform(lambda c: (c - c.mean()) / c.std()).astype(float).fillna(0).values
month_end["flag_if"] = IsolationForest(n_estimators=300, contamination=0.05, random_state=0).fit_predict(X) == -1
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05); month_end["flag_lof"] = lof.fit_predict(X) == -1
month_end["lof_score"] = -lof.negative_outlier_factor_
# DBSCAN : eps choisi sur la courbe des k-distances (quantile 90 % des distances au 5e voisin) ; le bruit (-1) = valeurs aberrantes
kdist = np.sort(NearestNeighbors(n_neighbors=5).fit(X).kneighbors(X)[0][:, -1])
eps = float(np.quantile(kdist, 0.90)); month_end["flag_dbscan"] = DBSCAN(eps=eps, min_samples=5).fit_predict(X) == -1
month_end["n_methodes"] = month_end[["flag_if", "flag_lof", "flag_dbscan"]].sum(axis=1)
print(f"eps DBSCAN = {eps:.2f} | IF : {month_end.flag_if.sum()} | LOF : {month_end.flag_lof.sum()} | DBSCAN : {month_end.flag_dbscan.sum()}")
print("Convergence (nb de méthodes) :", month_end.n_methodes.value_counts().sort_index().to_dict())
suspects = month_end[month_end.n_methodes >= 2].sort_values("lof_score", ascending=False)
print(suspects[cols + ["lof_score", "n_methodes"]].round(2).head(15).to_string(index=False))
print("\nRépartition des suspects par site / catégorie :")
print(pd.crosstab(suspects.facility_code, suspects.item_category))

eps DBSCAN = 1.80 | IF : 26 | LOF : 26 | DBSCAN : 38
Convergence (nb de méthodes) : {0: 455, 1: 17, 2: 17, 3: 13}
facility_code        item_category reporting_month  opening_stock  received_quantity  outgoing_quantity  closing_stock  ecart  z_mod inventory_status  lof_score  n_methodes
     FCL-1010 protective_equipment      2022-02-01           7.57               6.17               58.0            0.0  44.37  24.69           closed       5.77           3
     FCL-1020 agricultural_product      2023-09-01          38.01                0.0               71.0            0.0  34.15  18.91           closed       4.75           3
     FCL-1020  medical_consumables      2022-02-01          18.69                0.0               42.0            0.0  24.17  10.24           closed       4.08           3
     FCL-1020 monitoring_equipment      2023-09-01          18.73               4.17               25.0            0.0   2.48   1.34           closed       2.82           3
     FCL-1019 agricul

**Lecture.** Isolation Forest (26), LOF (26) et DBSCAN (38) sont croisés : 13 lignes sont signalées par les trois méthodes, 17 par deux. Les premières sont des mois où la **sortie déclarée dépasse largement le stock disponible** (ex. `FCL-1010` protective_equipment : 58 sortis pour 13,7 disponibles ; `FCL-1020` agricultural_product : 71 pour 38 ; `FCL-1020` medical_consumables : 42 pour 18,7), puis des baisses de stock non documentées sur `FCL-1010` (medical_consumables : −17,8 et −13,6) et `FCL-1019` (biologique, monitoring).
La standardisation est faite **par catégorie d'article** pour que les gros stocks (pallets) ne dominent pas mécaniquement la détection.

## 5. Croiser stocks, expéditions et ajustements (logistique)
**Question :** les mois où un site expédie réellement (expédition déclarée) ont-ils des écarts de stock plus forts que les autres mois ?
Et les ajustements d'inventaire comptabilisés expliquent-ils ces écarts ?

In [16]:
shipments["mois"] = shipments.dispatch_time.dt.to_period("M").dt.to_timestamp()
exped = shipments.groupby(["origin_facility_code", "item_category", "mois"]).declared_quantity.sum().rename("qte_expediee").reset_index()
me = month_end.merge(exped, left_on=["facility_code", "item_category", "reporting_month"],
                     right_on=["origin_facility_code", "item_category", "mois"], how="left")
me["mois_avec_expedition"] = me.qte_expediee.notna()
print("Mois-site-catégorie avec expédition déclarée retrouvée :", int(me.mois_avec_expedition.sum()), "/", len(me))
print("Sorties du stock = quantités expédiées sur ces mois : corrélation =",
      round(me[["outgoing_quantity", "qte_expediee"]].dropna().corr().iloc[0, 1], 3))
bio = me[me.item_category == "biological_sample"]
a, b = bio[bio.mois_avec_expedition].ecart, bio[~bio.mois_avec_expedition].ecart
print(f"\nÉchantillons biologiques – écart moyen avec expédition : {a.mean():.2f} (n={len(a)}) | sans : {b.mean():.2f} (n={len(b)}) | Mann-Whitney p = {stats.mannwhitneyu(a, b).pvalue:.5f}")
print(bio[bio.mois_avec_expedition].groupby("facility_code").ecart.agg(["count", "mean", "min"]).round(2))
# Sorties supérieures au stock disponible (stock + réceptions) = impossible physiquement
me["sortie_superieure_au_stock"] = me.outgoing_quantity > (me.opening_stock + me.received_quantity)
imp = me[me.sortie_superieure_au_stock & me.mois_avec_expedition]
print("\nExpéditions déclarées supérieures au stock disponible :")
print(imp[["facility_code", "item_category", "reporting_month", "opening_stock", "received_quantity", "outgoing_quantity", "closing_stock"]].round(2).to_string(index=False))

# Les ajustements comptabilisés (même site, même catégorie, ±1 mois) couvrent-ils les écarts négatifs importants ?
adjustments["mois"] = adjustments.adjustment_date.dt.to_period("M").dt.to_timestamp()
gros = me[(me.ecart < -4) & (me.item_category == "biological_sample")].copy()
def ajust_associes(r):
    m = adjustments[(adjustments.facility_code == r.facility_code) & (adjustments.item_category == r.item_category) &
                    adjustments.mois.between(r.reporting_month - pd.DateOffset(months=1), r.reporting_month + pd.DateOffset(months=1))]
    return pd.Series({"ajust_total": m.adjustment_quantity.sum(), "n_ajust": len(m), "motifs": "; ".join(sorted(set(m.adjustment_reason)))})
gros = pd.concat([gros[["facility_code", "reporting_month", "ecart"]], gros.apply(ajust_associes, axis=1)], axis=1)
gros["part_expliquee"] = (gros.ajust_total / gros.ecart).clip(0, 1)
print(f"\nÉcarts biologiques < -4 : {len(gros)} cas | écart cumulé {gros.ecart.sum():.1f} | ajustements associés {gros.ajust_total.sum():.1f} "
      f"| part moyenne expliquée {gros.part_expliquee.mean():.0%}")
print(gros.round(2).to_string(index=False))

Mois-site-catégorie avec expédition déclarée retrouvée : 27 / 502
Sorties du stock = quantités expédiées sur ces mois : corrélation = 1.0

Échantillons biologiques – écart moyen avec expédition : -7.24 (n=14) | sans : 0.11 (n=86) | Mann-Whitney p = 0.00013
               count  mean    min
facility_code                    
FCL-1010           1  2.03   2.03
FCL-1019           8 -7.30 -21.80
FCL-1020           5 -9.00 -12.36

Expéditions déclarées supérieures au stock disponible :
facility_code        item_category reporting_month  opening_stock  received_quantity  outgoing_quantity  closing_stock
     FCL-1019 agricultural_product      2021-05-01          15.18                0.0               17.0            0.0
     FCL-1020  medical_consumables      2022-02-01          18.69                0.0               42.0            0.0
     FCL-1020 monitoring_equipment      2023-09-01          18.73               4.17               25.0            0.0
     FCL-1020 agricultural_product      

**Lecture.** C'est le résultat central du volet stocks :
* Sur 27 mois-site-catégorie, la sortie comptable est exactement égale à la quantité expédiée déclarée (corrélation 1,0) : les deux tables décrivent bien les mêmes mouvements.
* Pour les échantillons biologiques, le stock baisse en moyenne de **−7,24** les mois avec expédition contre **+0,11** sans (Mann-Whitney p = 0,0001), sur `FCL-1019` (8 mois, −7,3) et `FCL-1020` (5 mois, −9,0). Autrement dit, **plus de matériel disparaît du stock que ce que l'expédition et l'usage documenté expliquent**.
* 5 expéditions déclarées dépassent le stock disponible (sorties jusqu'à environ 4× le stock disponible).
* Pour les 18 baisses biologiques < −4 : écart cumulé −175,1 contre −33,1 d'ajustements comptabilisés autour de la date, soit **environ 20 % expliqués**, presque toujours par « transfer posting correction » / « cycle count ». Six cas n'ont **aucun** ajustement (les cinq de `FCL-1027` et un de `FCL-1020`).
La continuité du stock (début = fin du mois précédent) est respectée dans 475 cas sur 477 : les mois manquants du fichier n'expliquent donc pas les écarts.
**Prudence :** c'est une *suspicion de perte ou de sortie non documentée*, pas une preuve de détournement (erreurs de comptage, pertes biologiques non saisies restent possibles).

## 6. Un flux atypique bien précis dans les expéditions
Les motifs « environmental and wildlife reference materials » et « temperature-controlled clinical programme delivery » sont rares et très structurés
(apparus dans les règles Apriori). On les compare au reste : expéditions « boucle » (origine = destination), acteurs, transporteur, et dossiers réglementaires liés.

In [18]:
print("Colonnes de facilities :")
print(facilities.columns.tolist())

print("\nContenu de facilities :")
display(facilities.head())

print("\nColonnes de bp :")
print(bp.columns.tolist())

Colonnes de facilities :
['facility_license_number', 'facility_name', 'facility_category', 'operating_organization_registration', 'street_address', 'region', 'latitude', 'longitude', 'license_start', 'license_end', 'facility_status', 'declared_services']

Contenu de facilities :


,facility_license_number,facility_name,facility_category,operating_organization_registration,street_address,region,latitude,longitude,license_start,license_end,facility_status,declared_services
0,FAC-VLN-101,Asterion Central Clinical Campus,research_hospital,REG-VLN-01002,"Asterion Central Clinical Campus, Valenne Central District",Valenne Republic,47.20,7.51,2001-02-01,NaN,active,preventive medicine; cardiac assessment; clinical research
1,FAC-AVL-201,Asterion Avelora Executive Clinic,private_clinic,REG-VLN-01002,"Asterion Avelora Executive Clinic, Eastern Avelora Agricultural Cl...",Avelora Federation,7.35,101.62,2008-05-15,NaN,active,executive health; specialist follow-up; outpatient procedures
2,FAC-AVL-301,Threshold Earth Avelora Laboratory,research_laboratory,REG-AVL-01007,"Threshold Earth Avelora Laboratory, Eastern Avelora Agricultural C...",Avelora Federation,7.50,101.75,2012-01-18,NaN,active,environmental health research; non-invasive human assessment



Colonnes de bp :
['partner_code', 'partner_name', 'partner_role', 'external_registration_number', 'billing_region', 'shipping_region', 'default_currency', 'payment_terms', 'partner_status', 'risk_class', 'onboarding_date']


In [19]:
shipments["boucle"] = shipments.origin_facility_code == shipments.destination_facility_code
M_ENV, M_CLIN = "environmental and wildlife reference materials", "temperature-controlled clinical programme delivery"
for motif in [M_ENV, M_CLIN]:
    s = shipments[shipments.declared_purpose == motif]
    print(f"\n=== {motif} : {len(s)} expéditions ({s.dispatch_time.min():%Y-%m} → {s.dispatch_time.max():%Y-%m})")
    for c in ["item_category", "sender_partner_code", "receiver_partner_code", "carrier_partner_code", "origin_facility_code", "destination_facility_code", "temperature_control_class"]:
        print(f"  {c:<26}", s[c].value_counts().to_dict())
tab = pd.crosstab(shipments.declared_purpose == M_ENV, shipments.boucle)
print("\nBoucles (origine = destination) :\n", tab, f"\nFisher exact p = {stats.fisher_exact(tab)[1]:.2e}")
names = pd.concat([
    bp.set_index("partner_code")["partner_name"],
    facilities.set_index("facility_license_number")["facility_name"]
])
# Dossiers réglementaires liés à des expéditions (SB05)
reg = pd.read_csv(glob.glob("**/regulatory_case_events.csv", recursive=True)[0])
cas = reg.dropna(subset=["related_shipment_number"]).drop_duplicates("case_number")[["case_number", "case_category", "subject_name", "related_shipment_number"]]
cas = cas.merge(shipments[["shipment_number", "item_category", "declared_purpose", "carrier_partner_code"]], left_on="related_shipment_number", right_on="shipment_number", how="left")
print(f"\nDossiers réglementaires rattachés à une expédition : {len(cas)} | dont motif « {M_ENV} » : {(cas.declared_purpose == M_ENV).sum()}")
part_env = (shipments.declared_purpose == M_ENV).mean()
print(f"Ce motif représente {part_env:.1%} des expéditions ; test binomial de sur-représentation p = {stats.binomtest(int((cas.declared_purpose == M_ENV).sum()), len(cas), part_env, alternative='greater').pvalue:.2e}")


=== environmental and wildlife reference materials : 18 expéditions (2018-02 → 2025-06)
  item_category              {'biological_sample': 18}
  sender_partner_code        {'BP-30056': 18}
  receiver_partner_code      {'BP-30078': 18}
  carrier_partner_code       {'BP-30047': 18}
  origin_facility_code       {'FCL-1030': 7, 'FCL-1019': 7, 'FCL-1020': 4}
  destination_facility_code  {'FCL-1024': 7, 'FCL-1019': 7, 'FCL-1020': 4}
  temperature_control_class  {'2-8c': 18}

=== temperature-controlled clinical programme delivery : 26 expéditions (2014-07 → 2024-02)
  item_category              {'medical_consumables': 14, 'biological_sample': 12}
  sender_partner_code        {'BP-30078': 26}
  receiver_partner_code      {'BP-30021': 26}
  carrier_partner_code       {'BP-30047': 26}
  origin_facility_code       {'FCL-1011': 26}
  destination_facility_code  {'FCL-1010': 14, 'FCL-1007': 12}
  temperature_control_class  {'2-8c': 26}

Boucles (origine = destination) :
 boucle            False  Tr

**Lecture.** Deux flux se détachent :
* *environmental and wildlife reference materials* : 18 envois, tous des échantillons biologiques 2-8 °C de `BP-30056` vers `BP-30078` (Meridian BioLogistics), toujours transportés par `BP-30047`, entre 2018 et 2025. **11 sur 18 sont des « boucles » (origine = destination)**, contre 11 sur 432 pour les autres envois (Fisher p ≈ 6·10⁻¹²) : un transport déclaré sans changement de site est atypique.
* *temperature-controlled clinical programme delivery* : 26 envois de Meridian (`FCL-1011`) vers les sites Asterion (`FCL-1010` : 14, `FCL-1007` : 12), 2014-2024, même transporteur.
Les dossiers réglementaires (SB05) rattachés à une expédition sont au nombre de 15 : **11 concernent le premier flux**, alors qu'il ne représente que 4 % des envois (p ≈ 5·10⁻¹³). Les régulateurs se sont donc intéressés à ce flux, ce qui corrobore le signal logistique de manière indépendante.

## 7. Réseau logistique : centralité et communautés (cours 4)
Graphe orienté des sites (arête = expéditions origine → destination, poids = nombre d'expéditions). Betweenness et closeness repèrent les sites « ponts » ;
Louvain regroupe les sites qui échangent davantage entre eux. (Les boucles sont comptées à part car elles n'apportent rien aux plus courts chemins.)

In [20]:
flux = shipments[~shipments.boucle].groupby(["origin_facility_code", "destination_facility_code"]).size().rename("w").reset_index()
G = nx.DiGraph(); G.add_weighted_edges_from(flux.itertuples(index=False, name=None), weight="w")
for u, v, d in G.edges(data=True): d["dist"] = 1 / d["w"]            # plus il y a de flux, plus la « distance » est courte
met = pd.DataFrame({"degre_sortant": dict(G.out_degree()), "degre_entrant": dict(G.in_degree()),
                    "force": dict(G.degree(weight="w")), "betweenness": nx.betweenness_centrality(G, weight="dist"),
                    "closeness": nx.closeness_centrality(G, distance="dist")})
met["nom"] = met.index.map(names); met["boucles"] = shipments[shipments.boucle].groupby("origin_facility_code").size().reindex(met.index).fillna(0).astype(int)
print(f"{G.number_of_nodes()} sites, {G.number_of_edges()} liens, densité = {nx.density(G):.3f}")
print(met.sort_values("betweenness", ascending=False).head(8).round(3).to_string())
comm = nx.community.louvain_communities(G.to_undirected(), weight="w", seed=0)
print("\nCommunautés Louvain (modularité = %.3f) :" % nx.community.modularity(G.to_undirected(), comm, weight="w"))
for i, c in enumerate(comm): print(f"  C{i} :", ", ".join(f"{x} ({names.get(x)})" for x in sorted(c)))

30 sites, 227 liens, densité = 0.261
          degre_sortant  degre_entrant  force  betweenness  closeness  nom  boucles
FCL-1011             15             19    146        0.380      1.372  NaN        2
FCL-1010             17             17     85        0.329      1.496  NaN        3
FCL-1019             11             12     73        0.236      1.353  NaN        7
FCL-1030             13             15     50        0.140      1.220  NaN        1
FCL-1020             17             10     43        0.117      1.099  NaN        4
FCL-1029             12             14     44        0.108      1.179  NaN        0
FCL-1007             10             11     75        0.072      1.365  NaN        0
FCL-1016             11             13     38        0.030      1.018  NaN        3

Communautés Louvain (modularité = 0.175) :
  C0 : FCL-1002 (None), FCL-1003 (None), FCL-1009 (None), FCL-1011 (None), FCL-1019 (None), FCL-1028 (None)
  C1 : FCL-1005 (None), FCL-1010 (None), FCL-1018 (None

**Lecture.** Le réseau de 30 sites est dense (0,26), sa structure en communautés est faible (modularité de Louvain 0,175) : il n'y a **pas de sous-réseau isolé**. En revanche, les sites les plus « ponts » (betweenness) sont `FCL-1011` Meridian Valenne HQ (0,38), `FCL-1010` Asterion Central Clinical Campus (0,33) et `FCL-1019` Meridian Avelora Transfer Annex (0,24) : les trois sites concernés par les flux et les écarts ci-dessus.

## 8. Lien avec les registres publics (`officer_filings`) – graphe dirigeants ↔ partenaires
Clé de jointure réelle : `officer_filings.registration_number` = `erp_business_partners.external_registration_number`. Pour une même personne,
`officer_local_ref` est identique d'un dossier à l'autre : on s'en sert pour distinguer un dirigeant commun d'un simple homonyme.

In [21]:
of = pd.read_csv(glob.glob("**/officer_filings.csv", recursive=True)[0])
of["full_name"] = of.full_name.str.strip().str.replace(r"\s+", " ", regex=True)
lien = of.merge(bp[["partner_code", "partner_name", "partner_role", "external_registration_number"]], left_on="registration_number", right_on="external_registration_number")
print(f"{lien.partner_code.nunique()} partenaires ERP retrouvés dans les registres publics ({len(lien)} mandats)")
B = nx.Graph(); B.add_edges_from((f"P:{r.officer_local_ref}", f"O:{r.partner_code}") for r in lien.itertuples())
persons = [n for n in B if n.startswith("P:")]
multi = [p for p in persons if B.degree(p) > 1]
print("Dirigeants présents dans plusieurs partenaires :", len(multi))
Gp = nx.Graph()
for p in multi:
    for a, b in itertools.combinations(sorted(x[2:] for x in B[p]), 2):
        Gp.add_edge(a, b, w=Gp[a][b]["w"] + 1 if Gp.has_edge(a, b) else 1)
poids = pd.DataFrame([(a, b, d["w"], names.get(a), names.get(b)) for a, b, d in Gp.edges(data=True)], columns=["A", "B", "dirigeants_communs", "nom_A", "nom_B"])
print(poids.sort_values("dirigeants_communs", ascending=False).head(8).to_string(index=False))
# Les partenaires liés par des dirigeants communs échangent-ils plus que le hasard ?
lies = {tuple(sorted(e)) for e in Gp.edges}
shipments["paire_exp_dest"] = [tuple(sorted(t)) for t in zip(shipments.sender_partner_code, shipments.receiver_partner_code)]
obs = int(shipments.paire_exp_dest.isin(lies).sum())
tous = list(itertools.combinations(sorted(lien.partner_code.unique()), 2)); rng = np.random.default_rng(1)
sim = [shipments.paire_exp_dest.isin({tous[i] for i in rng.choice(len(tous), len(lies), replace=False)}).sum() for _ in range(1000)]
print(f"\nExpéditions entre deux partenaires à dirigeant commun : {obs} (attendu au hasard ≈ {np.mean(sim):.1f}, p = {(np.sum(np.array(sim) >= obs) + 1) / 1001:.2f})")
comm_p = nx.community.louvain_communities(Gp, weight="w", seed=0)
print("Communautés de partenaires liés par des dirigeants :")
for c in comm_p: print("  ", [names.get(x) for x in sorted(c)])

24 partenaires ERP retrouvés dans les registres publics (136 mandats)
Dirigeants présents dans plusieurs partenaires : 27
       A        B  dirigeants_communs                              nom_A                              nom_B
BP-30039 BP-30078                   4 Valenne Industrial Futures Council              Meridian BioLogistics
BP-30004 BP-30017                   4        Common Ground Climate Forum       Meridian–Asterion Foundation
BP-30062 BP-30007                   4               Solvane Energy Group         Helix Legal and Compliance
BP-30042 BP-30043                   4          Nivren Maritime Logistics      Whitehaven Strategic Holdings
BP-30047 BP-30040                   4      Northstar Cold Chain Services        Threshold Earth Observatory
BP-30035 BP-30056                   4               Ardena Capital Union Horizon Field Research Cooperative
BP-30039 BP-30062                   1 Valenne Industrial Futures Council               Solvane Energy Group
BP-30042 BP-30

**Lecture.** 24 partenaires ERP sont retrouvés dans les registres publics ; 27 dirigeants siègent dans plusieurs partenaires. Six paires de partenaires partagent chacune 4 dirigeants, dont : Northstar Cold Chain Services ↔ Threshold Earth Observatory, Meridian BioLogistics ↔ Valenne Industrial Futures Council, Horizon Field Research Cooperative ↔ Ardena Capital Union (Horizon et Northstar sont l'expéditeur et le transporteur du flux environnemental).
**Résultat négatif à assumer :** ces partenaires liés par des dirigeants n'échangent pas davantage d'expéditions que le hasard (1 observée contre ≈ 3,9 attendue, p = 0,62). Les liens sont donc de gouvernance, pas commerciaux ; ils méritent un examen (conflits d'intérêts possibles), pas une conclusion.

## 9. Lien avec les données de santé (`clinical_observations`, `patient_administration`)
Les clés des deux univers ne se recoupent pas (codes `FCL-…` vs licences `FAC-…`). Hypothèse de rapprochement **par région** (à confirmer avec `licensed_facilities.csv`) :
`FCL-1010` (Valenne Central District) ↔ `FAC-VLN-101` ; `FCL-1007` (Eastern Avelora) ↔ `FAC-AVL-201/301`.
Test : les mois où des livraisons cliniques arrivent sur un site coïncident-ils avec davantage d'observations cardiaques « discordantes » ? (corrélation de Spearman + test de permutation par décalage circulaire).

In [22]:
pa = pd.read_csv(glob.glob("**/patient_administration.csv", recursive=True)[0])
ob = pd.read_csv(glob.glob("**/clinical_observations.csv", recursive=True)[0])
ob["t"] = pd.to_datetime(ob.observation_time); ob["mois"] = ob.t.dt.to_period("M").dt.to_timestamp()
ob["interpretation_code"] = norm_text(ob.interpretation_code)
ob = ob.merge(pa[["patient_number", "primary_facility_license"]].drop_duplicates("patient_number"), on="patient_number", how="left")
discord = ob[ob.interpretation_code == "discordant"]
print(f"{len(discord)} observations discordantes ({discord.observation_name.unique().tolist()}), {discord.patient_number.nunique()} patients")
clin = shipments[shipments.declared_purpose == M_CLIN].copy(); clin["mois"] = clin.dispatch_time.dt.to_period("M").dt.to_timestamp()
idx = pd.date_range("2014-01-01", "2026-01-01", freq="MS"); rng = np.random.default_rng(0); res = []
for nom, fcl, lic in [("Valenne (FCL-1010)", "FCL-1010", ["FAC-VLN-101"]), ("Avelora (FCL-1007)", "FCL-1007", ["FAC-AVL-201", "FAC-AVL-301"])]:
    s = clin[clin.destination_facility_code == fcl].groupby("mois").size().reindex(idx, fill_value=0)
    y = discord[discord.primary_facility_license.isin(lic)].groupby("mois").observation_number.nunique().reindex(idx, fill_value=0)
    rho = stats.spearmanr(s, y)[0]
    perm = [stats.spearmanr(np.roll(s.values, k), y)[0] for k in rng.integers(6, len(s) - 6, 1000)]
    res.append({"site": nom, "livraisons_cliniques": int(s.sum()), "rho_spearman": round(rho, 3), "p_permutation": round((np.sum(np.abs(perm) >= abs(rho)) + 1) / 1001, 4)})
print(pd.DataFrame(res).to_string(index=False))
print("Remarque : 2 tests réalisés → seuil de Bonferroni 0.025 ; résultat exploratoire (rapprochement de sites supposé).")

109 observations discordantes (['Apical cardiac rate']), 35 patients
              site  livraisons_cliniques  rho_spearman  p_permutation
Valenne (FCL-1010)                    14         0.054         0.4985
Avelora (FCL-1007)                    12         0.214         0.0070
Remarque : 2 tests réalisés → seuil de Bonferroni 0.025 ; résultat exploratoire (rapprochement de sites supposé).


**Lecture.** Les 109 observations discordantes concernent toutes la fréquence cardiaque apicale (35 patients). Pour le site Avelora, plus il y a de livraisons cliniques un mois donné, plus il y a d'observations discordantes (ρ = 0,21, p de permutation = 0,007, soit sous le seuil de Bonferroni de 0,025) ; pour Valenne il n'y a aucun lien (ρ = 0,05).
**Limites :** rapprochement de sites par région (hypothèse), seulement 12 livraisons, corrélation ≠ causalité. À présenter comme piste exploratoire.

## 10. Contrôles d'intégrité et qualité (réception, ajustements)

In [23]:
g = receipts.merge(po_lines, on=["purchase_order_number", "purchase_order_line"], how="left", suffixes=("", "_po"))
g["ratio_recu"] = g.received_quantity / g.ordered_quantity
print(f"Réceptions rattachées à une ligne de commande : {g.item_category.notna().sum()}/{len(g)} | fournisseur ≠ commande : {(g.supplier_partner_code != g.supplier_partner_code_po).sum()}")
print(f"Livraisons incomplètes (reçu < 90 % du commandé) : {(g.ratio_recu < 0.9).sum()} | jamais > commandé : {(g.ratio_recu <= 1).all()}")
print("Commandes sans réception :", int((~po_lines.purchase_order_number.isin(receipts.purchase_order_number)).sum()), po_lines[~po_lines.purchase_order_number.isin(receipts.purchase_order_number)].order_status.value_counts().to_dict())
# Les ajustements pointent-ils vers un reçu / une expédition cohérents ?
a_r = adjustments.dropna(subset=["related_receipt_number"]).merge(receipts[["receipt_number", "receiving_facility_code", "purchase_order_number", "purchase_order_line"]], left_on="related_receipt_number", right_on="receipt_number", how="left") \
        .merge(po_lines[["purchase_order_number", "purchase_order_line", "item_category"]], on=["purchase_order_number", "purchase_order_line"], how="left", suffixes=("", "_po"))
a_s = adjustments.dropna(subset=["related_shipment_number"]).merge(shipments[["shipment_number", "origin_facility_code", "destination_facility_code", "item_category"]], left_on="related_shipment_number", right_on="shipment_number", how="left", suffixes=("", "_sh"))
print(f"Ajustements liés à un reçu : {len(a_r)} → site différent : {(a_r.facility_code != a_r.receiving_facility_code).sum()}, catégorie différente : {(a_r.item_category != a_r.item_category_po).sum()}")
print(f"Ajustements liés à une expédition : {len(a_s)} → catégorie différente : {(a_s.item_category != a_s.item_category_sh).sum()}, site ni origine ni destination : {(~((a_s.facility_code == a_s.origin_facility_code) | (a_s.facility_code == a_s.destination_facility_code))).sum()}")
print("\nStatut des ajustements :", adjustments.adjustment_status.value_counts().to_dict())
print(adjustments.groupby("adjustment_status").adjustment_quantity.agg(["count", "sum"]).round(1))

Réceptions rattachées à une ligne de commande : 610/610 | fournisseur ≠ commande : 0
Livraisons incomplètes (reçu < 90 % du commandé) : 40 | jamais > commandé : True
Commandes sans réception : 40 {'delayed': 18, 'pending': 14, 'cancelled': 8}
Ajustements liés à un reçu : 23 → site différent : 21, catégorie différente : 22
Ajustements liés à une expédition : 37 → catégorie différente : 17, site ni origine ni destination : 18

Statut des ajustements : {'posted': 90, 'pending_review': 25, 'approved': 25}
                   count    sum
adjustment_status              
approved              25   51.4
pending_review        25    0.9
posted                90 -136.1


**Lecture.** Les réceptions sont cohérentes avec les commandes (fournisseur, site et unité identiques ; jamais plus que commandé ; 40 livraisons à moins de 90 %). Les 40 commandes sans réception sont toutes `delayed`/`pending`/`cancelled` : rien d'anormal. En revanche, les **liens des ajustements d'inventaire sont peu fiables** : 21 ajustements sur 23 rattachés à un reçu pointent vers un autre site et 22 sur 23 vers une autre catégorie ; 17 sur 37 rattachés à une expédition ont une catégorie différente. Ce sont des incohérences de traçabilité à signaler (justificatifs non vérifiables), sans qu'on puisse en conclure à une intention.